# Решения: DP 2D

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import csv


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


def load_coin_cases() -> list[dict[str, object]]:
    path = _find('coin_change_cases.csv')
    rows: list[dict[str, object]] = []
    with path.open(encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append({
                'case_id': row['case_id'],
                'amount': int(row['amount']),
                'coins': [int(x) for x in row['coins'].split()],
                'expected_min_coins': int(row['expected_min_coins']),
            })
    return rows


def load_grid() -> list[list[int]]:
    path = _find('route_cost_grid_4x5.csv')
    grid: list[list[int]] = []
    with path.open(encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            grid.append([int(x) for x in row])
    return grid


In [ ]:
def min_path_cost(grid: list[list[int]]) -> int:
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for c in range(1, cols):
        dp[0][c] = dp[0][c - 1] + grid[0][c]
    for r in range(1, rows):
        dp[r][0] = dp[r - 1][0] + grid[r][0]
    for r in range(1, rows):
        for c in range(1, cols):
            dp[r][c] = min(dp[r - 1][c], dp[r][c - 1]) + grid[r][c]
    return dp[-1][-1]

def count_paths(rows: int, cols: int) -> int:
    dp = [[0] * cols for _ in range(rows)]
    for r in range(rows):
        dp[r][0] = 1
    for c in range(cols):
        dp[0][c] = 1
    for r in range(1, rows):
        for c in range(1, cols):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    return dp[-1][-1]

def count_paths_with_blocks(blocks: set[tuple[int, int]], rows: int, cols: int) -> int:
    dp = [[0] * cols for _ in range(rows)]
    if (0, 0) in blocks:
        return 0
    dp[0][0] = 1
    for r in range(rows):
        for c in range(cols):
            if (r, c) in blocks:
                dp[r][c] = 0
                continue
            if r == 0 and c == 0:
                continue
            top = dp[r - 1][c] if r > 0 else 0
            left = dp[r][c - 1] if c > 0 else 0
            dp[r][c] = top + left
    return dp[-1][-1]

grid = load_grid()
assert min_path_cost(grid) == 11
assert count_paths(4, 5) == 35
blocks = {(1, 1), (2, 3)}
assert count_paths_with_blocks(blocks, 4, 5) == 7
BORDER_NOTE = (
    'Границы таблицы задают базовые случаи DP 2D: первая строка и первый столбец. '
    'Без корректных базовых ячеек переходы внутри таблицы дают неверный результат.'
)
print('grid=', grid)
print('min_path_cost=', min_path_cost(grid))
print('count_paths=', count_paths(4, 5))
print('count_paths_with_blocks=', count_paths_with_blocks(blocks, 4, 5))
print(BORDER_NOTE)